# 03 · CLIP — Contrastive Language-Image Pretraining

> **Source notes:** `CLIP.md`

How do image and text embeddings end up in the same space?

This notebook builds it from scratch and runs real CLIP:
- **InfoNCE loss** — implement the contrastive objective on a toy dataset
- **Similarity matrix** — visualise how CLIP scores image-text pairs
- **Zero-shot classification** — classify images without training
- **PixelSmith v2** — build a semantic image search engine using `sentence-transformers`

**All local. No GPU needed. No API key.** Runtime: < 2 minutes.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install",
                "torch", "torchvision", "sentence-transformers",
                "matplotlib", "numpy", "Pillow", "-q"], check=True)
print("Packages ready.")

## 1 · InfoNCE Loss — From Scratch

The InfoNCE loss treats finding the matching text for each image as an $N$-way classification problem.

- **Positives:** diagonal entries of the similarity matrix (matching pairs)
- **Negatives:** off-diagonal entries (mismatched pairs)
- **Temperature $\tau$:** sharpens the softmax; CLIP learns $\tau \approx 0.07$

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt


def infonce_loss(image_embeds: torch.Tensor, text_embeds: torch.Tensor,
                 temperature: float = 0.07) -> torch.Tensor:
    """
    Symmetric InfoNCE (contrastive) loss for CLIP training.
    Both inputs must be L2-normalised, shape (N, d).
    """
    N = image_embeds.shape[0]
    
    # Similarity matrix: (N, N) — scaled dot product (= cosine sim since L2 normed)
    logits = (image_embeds @ text_embeds.T) / temperature  # (N, N)
    
    # Ground truth: each image i matches text i → target is class index i
    labels = torch.arange(N)
    
    # Symmetric: image→text and text→image
    loss_img2txt = F.cross_entropy(logits,   labels)  # rows
    loss_txt2img = F.cross_entropy(logits.T, labels)  # columns
    
    return (loss_img2txt + loss_txt2img) / 2.0


# ── Demonstrate: random embeddings vs. well-aligned embeddings ───────────────
N, D = 8, 64  # 8 pairs, 64-dim embeddings

# Case 1: Untrained — random embeddings, no alignment
img_rand = F.normalize(torch.randn(N, D), dim=-1)
txt_rand = F.normalize(torch.randn(N, D), dim=-1)
loss_rand = infonce_loss(img_rand, txt_rand)

# Case 2: Perfectly aligned — matching pairs are identical, mismatches are orthogonal
base = F.normalize(torch.randn(N, D), dim=-1)
noise = F.normalize(torch.randn(N, D), dim=-1) * 0.02  # tiny noise
img_good = F.normalize(base + noise, dim=-1)
txt_good = F.normalize(base - noise, dim=-1)  # very similar to img_good
loss_good = infonce_loss(img_good, txt_good)

print(f"InfoNCE loss — untrained (random):   {loss_rand.item():.4f}  (≈ log(N) = {np.log(N):.4f})")
print(f"InfoNCE loss — well aligned:          {loss_good.item():.4f}  (close to 0 = perfect)")
print(f"\nUntrained model has maximum-entropy loss (uniform over N classes).")
print(f"Training minimises this by pushing matching pairs' similarity to 1.0.")

In [ ]:
# Visualise the similarity matrix for both cases
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, img_e, txt_e, title, loss_val in [
    (axes[0], img_rand, txt_rand, "Untrained: random embeddings", loss_rand),
    (axes[1], img_good, txt_good, "Well-aligned embeddings", loss_good),
]:
    sim = (img_e @ txt_e.T).detach().numpy()
    im = ax.imshow(sim, cmap="RdYlGn", vmin=-1, vmax=1, aspect="auto")
    ax.set_xlabel("Text index")
    ax.set_ylabel("Image index")
    ax.set_title(f"{title}\nInfoNCE loss = {loss_val.item():.4f}")
    plt.colorbar(im, ax=ax, label="Cosine similarity")
    # Highlight diagonal
    for i in range(N):
        ax.add_patch(plt.Rectangle((i-0.5, i-0.5), 1, 1, fill=False, edgecolor="blue", lw=2))

plt.suptitle("CLIP InfoNCE: Similarity matrix should be an identity-like pattern", y=1.02)
plt.tight_layout()
plt.show()

print("Blue squares = matching pairs (diagonal). These should be bright green.")
print("Off-diagonal entries should be near 0 (red/yellow = misaligned).")

## 2 · Temperature Effect

CLIP's temperature $\tau$ is a **learnable scalar**. Low $\tau$ → sharper softmax → harder task → better alignment.

In [ ]:
# Show how temperature affects the loss landscape
temperatures = [1.0, 0.5, 0.1, 0.07, 0.01]
results = []

for tau in temperatures:
    # Perfectly aligned pairs
    loss_aligned = infonce_loss(img_good, txt_good, temperature=tau)
    # Random pairs
    loss_random  = infonce_loss(img_rand, txt_rand, temperature=tau)
    results.append((tau, loss_aligned.item(), loss_random.item()))

print(f"{'Temperature':>12} {'Loss (aligned)':>16} {'Loss (random)':>14} {'Gap':>8}")
print("-" * 55)
for tau, la, lr in results:
    print(f"  τ = {tau:<8.3f}  {la:>14.4f}  {lr:>13.4f}  {lr - la:>7.4f}")

print("\nKey observation: lower τ → larger gap between aligned and random loss.")
print("This creates stronger gradient signal for the model to learn from.")
print("CLIP uses τ ≈ 0.07 (learned from data).")

## 3 · Real CLIP — Zero-Shot Classification

Using `sentence-transformers` with the `clip-ViT-B-32` model (same as OpenAI CLIP-B/32).

In [ ]:
from sentence_transformers import SentenceTransformer
from PIL import Image
import io, urllib.request

# Load CLIP model (~350 MB download on first run)
print("Loading CLIP ViT-B/32...")
clip_model = SentenceTransformer("clip-ViT-B-32")
print("Model loaded.")

# ── Zero-shot classification ──────────────────────────────────────────────────
# Class names → prompt templates (following OpenAI CLIP paper recommendation)
class_names = ["cat", "dog", "airplane", "car", "bird", "fish", "tree", "mountain"]
prompts = [f"a photo of a {c}" for c in class_names]

# Encode class prompts → class text embeddings
print("Encoding class prompts...")
text_embeddings = clip_model.encode(prompts, convert_to_tensor=True, normalize_embeddings=True)
print(f"Text embeddings: {text_embeddings.shape}  ({len(class_names)} classes × 512 dims)")

In [ ]:
# Try to download some test images; fallback to synthetic if no network
test_images = []
test_labels = []

image_urls = [
    ("cat",      "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/320px-Cat03.jpg"),
    ("dog",      "https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/320px-YellowLabradorLooking_new.jpg"),
    ("airplane", "https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/320px-PNG_transparency_demonstration_1.png"),
]

for label, url in image_urls:
    try:
        with urllib.request.urlopen(url, timeout=5) as r:
            img = Image.open(io.BytesIO(r.read())).convert("RGB")
        test_images.append(img)
        test_labels.append(label)
        print(f"  Downloaded: {label}")
    except Exception as e:
        print(f"  Skipped {label} (network unavailable): {e}")

if not test_images:
    # Fallback: create simple synthetic images with distinct colour patterns
    print("Creating synthetic test images (network unavailable)...")
    colours = [(200, 100, 50), (50, 150, 200), (100, 200, 80)]
    fallback_labels = ["orange image", "blue image", "green image"]
    for colour, lbl in zip(colours, fallback_labels):
        arr = np.full((224, 224, 3), colour, dtype=np.uint8)
        test_images.append(Image.fromarray(arr))
        test_labels.append(lbl)

print(f"\nTest images loaded: {len(test_images)}")

In [ ]:
# Encode test images and compute similarity to each class
print("Encoding test images...")
image_embeddings = clip_model.encode(test_images, convert_to_tensor=True, normalize_embeddings=True)
print(f"Image embeddings: {image_embeddings.shape}")

# Similarity: (n_images, n_classes)
similarity = (image_embeddings @ text_embeddings.T).cpu().numpy()

print(f"\n{'Image (true label)':<22} {'Predicted':>10} {'Confidence':>12} {'Correct?':>9}")
print("-" * 57)
for i, true_label in enumerate(test_labels):
    scores      = similarity[i]
    probs       = np.exp(scores / 0.07) / np.exp(scores / 0.07).sum()  # softmax @ τ=0.07
    predicted   = class_names[scores.argmax()]
    confidence  = probs.max()
    is_correct  = "✓" if predicted.lower() in true_label.lower() else "✗"
    print(f"  {true_label:<20} {predicted:>10} {confidence:>12.1%} {is_correct:>9}")

In [ ]:
# Visualise similarity scores
if test_images:
    n_imgs = len(test_images)
    fig, axes = plt.subplots(1, n_imgs, figsize=(5 * n_imgs, 4))
    if n_imgs == 1:
        axes = [axes]
    
    for ax, img, label, scores in zip(axes, test_images, test_labels, similarity):
        # Softmax for probabilities
        probs = np.exp(scores / 0.07) / np.exp(scores / 0.07).sum()
        colours = ["green" if c == class_names[scores.argmax()] else "steelblue" for c in class_names]
        ax.barh(class_names, probs, color=colours)
        ax.set_xlim(0, 1)
        ax.set_xlabel("Probability")
        ax.set_title(f"True: '{label}'")
        ax.invert_yaxis()
    
    plt.suptitle("CLIP Zero-Shot Classification\nGreen = predicted class", y=1.03)
    plt.tight_layout()
    plt.show()

## 4 · PixelSmith v2 — Semantic Image Search

A text query retrieves the most semantically similar images from a collection — without any keyword matching.

In [ ]:
# Build a small image index
# In production: 1M+ images; here we use synthetic images with descriptive contexts

# Simulate an image collection with known semantic content
# (since we can't download many images, we'll use text-only to demonstrate the principle)
print("=" * 60)
print("PixelSmith v2 — Text-to-Image Semantic Search")
print("=" * 60)

# A small corpus of image descriptions (simulating what the images contain)
image_corpus = [
    "a golden retriever running on the beach",
    "a tabby cat sleeping on a windowsill",
    "a red sports car on a mountain road",
    "a steam locomotive at a train station",
    "a bowl of fresh strawberries",
    "an astronaut floating in space",
    "a medieval castle at sunset",
    "a crowded city street at night with neon signs",
    "a whale breaching the ocean surface",
    "a child flying a kite in a green field",
]

# Encode all 'images' (using text descriptions as proxies)
corpus_embeddings = clip_model.encode(image_corpus, convert_to_tensor=True, normalize_embeddings=True)
print(f"Index built: {len(image_corpus)} images, embedding shape: {corpus_embeddings.shape}")

# Search with text queries
queries = [
    "a dog at the seaside",
    "vintage transportation",
    "urban nightlife",
    "something in the ocean",
]

query_embeddings = clip_model.encode(queries, convert_to_tensor=True, normalize_embeddings=True)

print(f"\n{'Query':<30} {'Rank 1 Result':<45} {'Score':>6}")
print("-" * 85)

query_sim = (query_embeddings @ corpus_embeddings.T).cpu().numpy()

for query, scores in zip(queries, query_sim):
    top_idx = scores.argsort()[::-1][:3]
    print(f"\n  Query: '{query}'")
    for rank, idx in enumerate(top_idx, 1):
        print(f"    #{rank}: {image_corpus[idx]:<42}  sim={scores[idx]:.4f}")

## 5 · CLIP as the Text Encoder in Stable Diffusion

CLIP's text encoder is used *frozen* inside Stable Diffusion. Understanding its output format is important for understanding how conditioning works in Chapter 5 (Guidance).

In [ ]:
# In Stable Diffusion, the text encoder produces a sequence of token embeddings
# (not just the final CLS pooled embedding), which are fed as cross-attention keys/values
# into the U-Net at every decoder layer.

# Here we show the shape difference between pooled vs. sequence CLIP output
sample_prompt = "a photograph of a mountain lake at golden hour, dramatic lighting"

# Standard CLIP output: 512-dim pooled embedding
pooled = clip_model.encode([sample_prompt], convert_to_tensor=True, normalize_embeddings=True)
print(f"Standard CLIP output (pooled):       {pooled.shape}")
print(f"  → Used for image search / zero-shot classification")
print(f"  → Single vector, loses per-token detail")

print()
print(f"Stable Diffusion text conditioning:  (1, 77, 768)")
print(f"  → 77 token positions (CLIP's fixed context length)")
print(f"  → 768-dim per token (ViT-L/14 hidden dim used in SD v2)")
print(f"  → Fed as cross-attention K and V to U-Net at every layer")
print(f"  → Model learns which image regions attend to which words")

print()
print(f"SDXL uses two text encoders concatenated:")
print(f"  OpenCLIP ViT-bigG:  (1, 77, 1280)")
print(f"  CLIP ViT-L/14:      (1, 77, 768)")
print(f"  Concatenated:       (1, 77, 2048)  ← richer conditioning")

## 6 · Summary — PixelSmith v2

```
┌──────────────────────────────────────────────────────────────────────────────┐
│ PixelSmith v2 — Semantic Image Search                                         │
│                                                                                │
│  Collection of images                                                          │
│      │  ViT image encoder → 512-dim L2-normed embeddings                     │
│      ▼                                                                         │
│  Index: { image_id → embedding }                                              │
│                                                                                │
│  Text query                                                                    │
│      │  CLIP text encoder → 512-dim L2-normed embedding                      │
│      ▼                                                                         │
│  dot product with every index entry → ranked results                          │
│                                                                                │
│  Zero training. Zero labels. Zero fine-tuning.                                │
│  Works because image and text live in the SAME embedding space.               │
└──────────────────────────────────────────────────────────────────────────────┘
```

**Next:** [DiffusionModels.md](../DiffusionModels/DiffusionModels.md) — to generate images (PixelSmith v3) we need a completely different mechanism. Diffusion models start from pure noise and iteratively denoise toward a plausible image. CLIP's text encoder will later slot in as the conditioning signal that guides this denoising process.